RetailPulse 360

Notebook 06 — Inventory Snapshots

Goal: sales.csv tells us what SOLD. It says nothing about what's currently SITTING on
shelves. This notebook generates a realistic current-stock-level snapshot for every
store-SKU pair — the missing piece that unlocks Inventory Turnover, Stock Alerts, and
eventually the redistribution engine (Phase 4).

Input: stores.csv, skus.csv, sales.csv
Output: inventory_snapshots.csv

In [1]:
# 1. IMPORTS
# ============================================================

import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# 2. LOAD INPUTS
# ============================================================

BASE_PATH = "/kaggle/input/datasets/hamaz911/notebook-6-dataset/"

stores = pd.read_csv(BASE_PATH + "stores.csv")
skus = pd.read_csv(BASE_PATH + "skus.csv")
sales = pd.read_csv(BASE_PATH + "sales.csv", parse_dates=["date"])

print("Loaded:")
print("  stores:", stores.shape)
print("  skus:", skus.shape)
print("  sales:", sales.shape)
print("  sales date range:", sales["date"].min(), "to", sales["date"].max())

Loaded:
  stores: (191, 18)
  skus: (3296, 13)
  sales: (2459464, 4)
  sales date range: 2024-02-23 00:00:00 to 2026-08-23 00:00:00


In [3]:
# 3. DATA QUALITY — VALIDATE INPUTS
# ============================================================

print("Missing values:")
for name, df in [("stores", stores), ("skus", skus), ("sales", sales)]:
    m = df.isna().sum()
    if m.sum() > 0:
        print(f"  {name}: {dict(m[m > 0])}")
print("(nothing printed above = no missing values)")

print("\nDuplicate sku_id in skus.csv:", skus["sku_id"].duplicated().sum())
print("Duplicate store_id in stores.csv:", stores["store_id"].duplicated().sum())

# Referential integrity: does every sale reference a real store and real SKU?
orphan_stores = ~sales["store_id"].isin(stores["store_id"])
orphan_skus = ~sales["sku_id"].isin(skus["sku_id"])
print("\nSales rows referencing a non-existent store:", orphan_stores.sum())
print("Sales rows referencing a non-existent SKU:", orphan_skus.sum())

# Sanity check on units_sold
print("\nunits_sold sanity check:")
print(sales["units_sold"].describe())
assert (sales["units_sold"] > 0).all(), "Found non-positive units_sold — investigate"

Missing values:
  stores: {'store_size': np.int64(1), 'rossmann_store_id': np.int64(1), 'dow_mon': np.int64(1), 'dow_tue': np.int64(1), 'dow_wed': np.int64(1), 'dow_thu': np.int64(1), 'dow_fri': np.int64(1), 'dow_sat': np.int64(1), 'dow_sun': np.int64(1), 'promo_lift': np.int64(1), 'trend_pct_per_year': np.int64(1), 'volatility_cv': np.int64(1), 'holiday_lift': np.int64(1)}
(nothing printed above = no missing values)

Duplicate sku_id in skus.csv: 0
Duplicate store_id in stores.csv: 0

Sales rows referencing a non-existent store: 0
Sales rows referencing a non-existent SKU: 0

units_sold sanity check:
count    2.459464e+06
mean     1.017655e+00
std      1.342172e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      4.000000e+00
Name: units_sold, dtype: float64


In [4]:
# 4. COMPUTE AVERAGE DAILY SALES VELOCITY PER STORE-SKU
# ============================================================
# "Velocity" = how many units, on average, a store-SKU sells per day.
# This drives how much stock we'd expect that store-SKU to be holding.

TOTAL_DAYS = (sales["date"].max() - sales["date"].min()).days + 1
print("Total days in the sales period:", TOTAL_DAYS)

velocity = sales.groupby(["store_id", "sku_id"])["units_sold"].sum().reset_index()
velocity["avg_daily_velocity"] = velocity["units_sold"] / TOTAL_DAYS
velocity = velocity.rename(columns={"units_sold": "total_units_sold_period"})

print("Store-SKU pairs with sales history:", len(velocity))
print("\nVelocity distribution:")
print(velocity["avg_daily_velocity"].describe())

Total days in the sales period: 913
Store-SKU pairs with sales history: 159151

Velocity distribution:
count    159151.000000
mean          0.017225
std           0.017474
min           0.001095
25%           0.004381
50%           0.010953
75%           0.024096
max           0.123768
Name: avg_daily_velocity, dtype: float64


In [5]:
# 5. REGENERATE THE TRUE ASSORTMENT (DETERMINISTIC, SAME SEED AS NOTEBOOK 05)
# ============================================================
# sales.csv only shows SKUs that sold at least once — using it alone would
# silently drop any assigned SKU that never sold, wrongly treating "dead
# stock" as "not assigned at all". Since the original assortment generation
# was seeded, we regenerate it exactly rather than approximate.

ASSORTMENT_FRACTION = {"Large": 0.65, "Medium": 0.40, "Small": 0.22}
np.random.seed(11)

physical_stores = stores[stores["channel"] == "Physical"].copy()
all_sku_ids = skus["sku_id"].values

assortment_rows = []
for _, store in physical_stores.iterrows():
    frac = ASSORTMENT_FRACTION[store["store_size"]]
    n_skus = int(len(all_sku_ids) * frac)
    assigned_skus = np.random.choice(all_sku_ids, size=n_skus, replace=False)
    for sku in assigned_skus:
        assortment_rows.append({"store_id": store["store_id"], "sku_id": sku})

assortment = pd.DataFrame(assortment_rows)
print("Regenerated assortment pairs:", len(assortment))
assert len(assortment) == 228724, "Doesn't match Notebook 05's original count — regeneration failed"
print("Matches Notebook 05's original 228,724 pairs \u2014 regeneration confirmed correct.")

Regenerated assortment pairs: 228724
Matches Notebook 05's original 228,724 pairs — regeneration confirmed correct.


In [6]:
# 6. MERGE ASSORTMENT + VELOCITY, COMPUTE CURRENT STOCK LEVEL
# ============================================================
# Zero-velocity SKUs get a small baseline (BASELINE_DEAD_STOCK) rather
# than 0 — a SKU that was stocked but never sold is real dead stock/
# overstock, not "doesn't exist". Erasing it to 0 would hide exactly the
# signal the redistribution engine (Phase 4) needs to find later.

TARGET_DAYS = {"Large": 28, "Medium": 21, "Small": 14}
BASELINE_DEAD_STOCK = 2  # documented assumption: minimum stocking quantity per SKU

inventory = assortment.merge(velocity, on=["store_id", "sku_id"], how="left")
inventory["avg_daily_velocity"] = inventory["avg_daily_velocity"].fillna(0)
inventory["total_units_sold_period"] = inventory["total_units_sold_period"].fillna(0)

inventory = inventory.merge(stores[["store_id", "store_size"]], on="store_id", how="left")
inventory["target_days"] = inventory["store_size"].map(TARGET_DAYS)

inventory["target_stock"] = inventory["avg_daily_velocity"] * inventory["target_days"]
inventory["target_stock"] = np.where(
    inventory["avg_daily_velocity"] == 0, BASELINE_DEAD_STOCK, inventory["target_stock"]
)

print("Total inventory rows:", len(inventory))
print("Zero-velocity (dead stock) rows:", (inventory["avg_daily_velocity"] == 0).sum())
print("\nTarget stock distribution:")
print(inventory["target_stock"].describe())

Total inventory rows: 228724
Zero-velocity (dead stock) rows: 69573

Target stock distribution:
count    228724.000000
mean          0.874887
std           0.820423
min           0.015334
25%           0.138007
50%           0.460022
75%           2.000000
max           2.821468
Name: target_stock, dtype: float64


In [7]:
# 6b. ROUND TO REALISTIC WHOLE UNITS
# ============================================================
# Real stores don't stock fractional shoes. Any SKU with actual sales
# velocity gets rounded up to at least 1 unit (you order in whole pairs,
# even for a slow seller) — dead-stock SKUs keep their flat baseline.

inventory["target_stock"] = np.ceil(inventory["target_stock"]).astype(int)
inventory["target_stock"] = np.where(
    (inventory["avg_daily_velocity"] > 0) & (inventory["target_stock"] < 1),
    1, inventory["target_stock"]
)

print("Target stock distribution (after rounding):")
print(inventory["target_stock"].describe())
print("\nValue counts (top 10):")
print(inventory["target_stock"].value_counts().sort_index().head(10))

Target stock distribution (after rounding):
count    228724.000000
mean          1.376366
std           0.488161
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max           3.000000
Name: target_stock, dtype: float64

Value counts (top 10):
target_stock
1    143050
2     85264
3       410
Name: count, dtype: int64


In [8]:
# 6c. CURRENT STOCK VIA POISSON (allows genuine stockouts, no artificial floor)
# ============================================================
# Deterministic ceil+floor-at-1 made a real stockout mathematically
# impossible for any SKU with real sales history — backwards, since
# detecting stockouts is the whole point of this project. Poisson
# sampling (same technique as Notebook 05's sales generation) naturally
# produces 0 (stockout), 1 (last pair), or more, based on how fast that
# SKU actually moves.

inventory["target_stock"] = inventory["avg_daily_velocity"] * inventory["target_days"]
inventory["target_stock"] = np.where(
    inventory["avg_daily_velocity"] == 0, BASELINE_DEAD_STOCK, inventory["target_stock"]
)

np.random.seed(15)
cycle_factor = np.random.uniform(0.4, 1.5, size=len(inventory))

# Dead-stock SKUs keep their deterministic baseline (they were stocked,
# never moved — no reason to model that as a random process).
# Real-velocity SKUs get Poisson-sampled stock.
is_dead = inventory["avg_daily_velocity"] == 0
lam = inventory["target_stock"] * cycle_factor
poisson_stock = np.random.poisson(lam)

inventory["current_stock"] = np.where(is_dead, BASELINE_DEAD_STOCK, poisson_stock)

print("current_stock distribution:")
print(inventory["current_stock"].describe())
print("\nGenuine stockouts (0 units, real-velocity SKUs):", 
      ((inventory["current_stock"] == 0) & ~is_dead).sum())
print("Value counts:")
print(inventory["current_stock"].value_counts().sort_index().head(10))

current_stock distribution:
count    228724.000000
mean          0.859468
std           0.973793
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max           8.000000
Name: current_stock, dtype: float64

Genuine stockouts (0 units, real-velocity SKUs): 119050
Value counts:
current_stock
0    119050
1     28196
2     77727
3      2556
4       821
5       288
6        67
7        15
8         4
Name: count, dtype: int64


In [9]:
# 7. AGGREGATE TO STORE+PRODUCT LEVEL — SEPARATE ACTIVE VS DEAD STOCK
# ============================================================
# Dead-stock SKUs (baseline units, zero velocity) were inflating days-of-
# supply for every product by adding stock without adding velocity to
# the denominator. Fixed: days_of_supply is computed from ACTIVE
# (real-velocity) SKUs only; dead stock is reported as its own metric.

inv_with_product = inventory.merge(skus[["sku_id", "product_id"]], on="sku_id", how="left")

active_skus = inv_with_product[inv_with_product["avg_daily_velocity"] > 0]
dead_skus = inv_with_product[inv_with_product["avg_daily_velocity"] == 0]

active_agg = active_skus.groupby(["store_id", "product_id"]).agg(
    active_stock=("current_stock", "sum"),
    active_daily_velocity=("avg_daily_velocity", "sum"),
).reset_index()
active_agg["days_of_supply"] = active_agg["active_stock"] / active_agg["active_daily_velocity"]

dead_agg = dead_skus.groupby(["store_id", "product_id"]).agg(
    dead_stock_units=("current_stock", "sum"),
).reset_index()

product_level = active_agg.merge(dead_agg, on=["store_id", "product_id"], how="outer")
product_level[["active_stock", "dead_stock_units"]] = product_level[["active_stock", "dead_stock_units"]].fillna(0)

product_level["stock_status"] = pd.cut(
    product_level["days_of_supply"],
    bins=[-0.01, 7, 21, 60, np.inf],
    labels=["Critical (Reorder)", "Low", "Healthy", "Overstock"]
)
# products with NO active SKUs at all (100% dead) have no days_of_supply -> its own status
product_level["stock_status"] = product_level["stock_status"].cat.add_categories(["Fully Dead"])
product_level.loc[product_level["days_of_supply"].isna(), "stock_status"] = "Fully Dead"

print("Store-product rows:", len(product_level))
print("\nStock status breakdown:")
print(product_level["stock_status"].value_counts())
print("\nDays of supply (active only) distribution:")
print(product_level["days_of_supply"].describe())
print("\nDead stock units — total sitting dead across all store-products:", product_level["dead_stock_units"].sum())

Store-product rows: 13678

Stock status breakdown:
stock_status
Healthy               5153
Low                   4890
Critical (Reorder)    3469
Overstock              157
Fully Dead               9
Name: count, dtype: int64

Days of supply (active only) distribution:
count    13669.000000
mean        17.951053
std         14.903113
min          0.000000
25%          6.762963
50%         17.557692
75%         26.179211
max        228.250000
Name: days_of_supply, dtype: float64

Dead stock units — total sitting dead across all store-products: 139146.0


In [10]:
# 8. SAVE OUTPUTS
# ============================================================
# Two granularities, two different jobs:
#   - inventory_snapshots.csv: SKU-level (matches Product Catalog page)
#   - inventory_turnover_summary.csv: store+product level (drives Home
#     page's Inventory Turnover / Stock Alerts cards)

inventory[["store_id", "sku_id", "avg_daily_velocity", "current_stock"]].to_csv(
    "inventory_snapshots.csv", index=False
)
print("Saved inventory_snapshots.csv —", inventory.shape[0], "rows")

product_level.to_csv("inventory_turnover_summary.csv", index=False)
print("Saved inventory_turnover_summary.csv —", product_level.shape[0], "rows")

Saved inventory_snapshots.csv — 228724 rows
Saved inventory_turnover_summary.csv — 13678 rows


In [11]:
# 9. NOTEBOOK SUMMARY
# ============================================================

print("NOTEBOOK 06 SUMMARY")
print(f"Store-SKU inventory rows: {len(inventory):,}")
print(f"Genuine stockouts (0 units): {((inventory['current_stock']==0) & (inventory['avg_daily_velocity']>0)).sum():,}")
print(f"Dead-stock SKUs (never sold): {(inventory['avg_daily_velocity']==0).sum():,}")
print(f"Store-product summary rows: {len(product_level):,}")
print("Stock status breakdown:")
print(product_level["stock_status"].value_counts().to_string())
print("\nMethodology: current stock modeled via Poisson process (same technique")
print("as Notebook 05's sales generation) driven by real sales velocity x")
print("store-size-based days-of-supply targets x depletion-cycle variance.")
print("Dead-stock SKUs (baseline units, never sold) reported separately from")
print("active stock health, avoiding a misleading blended metric.")
print("\nOutput files: inventory_snapshots.csv, inventory_turnover_summary.csv")
print("\n\u2713 Notebook 06 completed successfully.")

NOTEBOOK 06 SUMMARY
Store-SKU inventory rows: 228,724
Genuine stockouts (0 units): 119,050
Dead-stock SKUs (never sold): 69,573
Store-product summary rows: 13,678
Stock status breakdown:
stock_status
Healthy               5153
Low                   4890
Critical (Reorder)    3469
Overstock              157
Fully Dead               9

Methodology: current stock modeled via Poisson process (same technique
as Notebook 05's sales generation) driven by real sales velocity x
store-size-based days-of-supply targets x depletion-cycle variance.
Dead-stock SKUs (baseline units, never sold) reported separately from
active stock health, avoiding a misleading blended metric.

Output files: inventory_snapshots.csv, inventory_turnover_summary.csv

✓ Notebook 06 completed successfully.
